In [ ]:
!pip install -q "transformers>=4.51.0" "datasets>=3.0.0" "accelerate>=1.0.0" huggingface_hub safetensors sentencepiece

In [ ]:
import os, re, json, math, time, random, shutil, subprocess, hashlib, sys, gc, zipfile, platform
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, Qwen3Config, Qwen3ForCausalLM
from torch.utils.data import Dataset as TorchDataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", platform.python_version(), "| PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda": print(torch.cuda.get_device_name(0), "| VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
else: raise RuntimeError("A CUDA GPU is required for this training configuration. In Colab select a GPU runtime.")

from google.colab import drive
drive.mount('/content/drive')
PROJECT_NAME = "Vibe_Coding_Qwen3_SLM_Project"
PROJECT_DIR = Path('/content/drive/MyDrive') / PROJECT_NAME
for folder in ['checkpoints','hf_model','hf_dataset','gguf','reports','logs','github_repo']:
    (PROJECT_DIR/folder).mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_DIR)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Python: 3.13.16 | PyTorch: 2.14.1+cu130
Device: cuda
Tesla T4 | VRAM GB: 14.56
Mounted at /content/drive
Project directory: /content/drive/MyDrive/Vibe_Coding_Qwen3_SLM_Project


## 1. Project configuration

Change the Hugging Face account/repository names to your own. Upload switches are deliberately explicit because they require your account credentials. Do not paste access tokens into notebook source cells.

In [ ]:
DATASET_ID = "CodeDevX/Vibe-Coding-Instruct"
TOKENIZER_ID = "Qwen/Qwen3-8B"
MODEL_REPO_ID = "Aslam7/Vibe-Coding-Qwen3-SLM"
DATASET_REPO_ID = "Aslam7/Vibe-Coding-Instruct-Prepared"
GITHUB_REPO_NAME = "Vibe-Coding-Qwen3-SLM"

MAX_STEPS = 1200                 # optimizer updates; fixed requested ceiling
BATCH_SIZE = 1                   # memory-safe on Colab T4
GRAD_ACCUM_STEPS = 8             # effective batch size = 8
MAX_LENGTH = 128                 # retained from the reference notebook
LEARNING_RATE = 2e-4
MIN_LR = 2e-5
WARMUP_STEPS = 60
EVAL_EVERY = 100
SAVE_EVERY = 200
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
RESUME_FROM_CHECKPOINT = False   # True only after an interrupted run
UPLOAD_TO_HUGGING_FACE = True   # set True after configuring repo IDs
CONVERT_TO_GGUF = True
PUSH_TO_GITHUB = False           # set True after configuring repo + token

CHECKPOINT_DIR = PROJECT_DIR/'checkpoints'
EXPORT_DIR = PROJECT_DIR/'hf_model'
DATASET_EXPORT_DIR = PROJECT_DIR/'hf_dataset'
print("Optimizer update ceiling:", MAX_STEPS, "| effective batch:", BATCH_SIZE*GRAD_ACCUM_STEPS)
print("No pretrained model weights will be loaded.")

Optimizer update ceiling: 1200 | effective batch: 8
No pretrained model weights will be loaded.


## 2. Load and audit the source dataset

We inspect the actual columns and remove repeated identical instruction/answer records. Scenario numbers are stripped with a correctly escaped regular expression. Rows are grouped by the normalized instruction + optional input so equivalent templates cannot leak across splits.

In [ ]:
raw = load_dataset(DATASET_ID)
source_split = 'train' if 'train' in raw else list(raw.keys())[0]
source = raw[source_split]
print(raw)
print("Source split:", source_split, "| rows:", len(source), "| columns:", source.column_names)
required = {'instruction','output'}
if not required.issubset(source.column_names):
    raise ValueError(f"Expected columns {required}; found {source.column_names}")

def clean_instruction(s):
    s = str(s or '').strip()
    s = re.sub(r'\bScenario\s+\d+\s*:?', ' ', s, flags=re.I)
    return re.sub(r'\s+', ' ', s).strip()

def normalize(s):
    return re.sub(r'\s+', ' ', str(s or '').strip()).casefold()

# Deduplicate exact semantic pairs. Retain one example for each unique instruction/input/output mapping.
unique = {}
for row in source:
    ins = clean_instruction(row.get('instruction',''))
    extra = str(row.get('input') or '').strip() if 'input' in source.column_names else ''
    ans = str(row.get('output') or '').strip()
    if not ins or not ans: continue
    group_key = normalize(ins) + '\u241f' + normalize(extra)
    pair_key = group_key + '\u241f' + normalize(ans)
    if pair_key not in unique:
        unique[pair_key] = {'instruction':ins, 'input':extra, 'output':ans, 'group_key':group_key}
records = list(unique.values())
group_keys = sorted({r['group_key'] for r in records})
print("Raw rows:", len(source))
print("Unique instruction/input/output pairs:", len(records))
print("Unique instruction/input templates:", len(group_keys))
print("Unique outputs:", len({normalize(r['output']) for r in records}))
if len(group_keys) < 30:
    print("\nDATASET WARNING: very low template diversity. Training can demonstrate pipeline mechanics but cannot support broad coding-generalization claims.")
if len(group_keys) < 3:
    raise ValueError("At least 3 unique instruction templates are needed for train/validation/test splits.")

# Save audit as a reproducibility artifact.
audit = {'dataset_id':DATASET_ID,'raw_rows':len(source),'unique_pairs':len(records),'unique_templates':len(group_keys),'unique_outputs':len({normalize(r['output']) for r in records}), 'deduplication':'exact normalized instruction/input/output', 'split_unit':'normalized instruction + input template'}
(PROJECT_DIR/'reports'/'dataset_audit.json').write_text(json.dumps(audit,indent=2))
pd.DataFrame([audit]).to_csv(PROJECT_DIR/'reports'/'dataset_audit.csv',index=False)

README.md:   0%|          | 0.00/176 [00:00<?, ?B/s]

dataset.jsonl: reconstructing file:   0%|          |  0.00B /  459MB            

dataset.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1100000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'prompt'],
        num_rows: 1100000
    })
})
Source split: train | rows: 1100000 | columns: ['instruction', 'input', 'output', 'prompt']
Raw rows: 1100000
Unique instruction/input/output pairs: 14
Unique instruction/input templates: 14
Unique outputs: 14

DATASET WARNING: very low template diversity. Training can demonstrate pipeline mechanics but cannot support broad coding-generalization claims.


## 3. Leakage-resistant train / validation / test split

The split is by normalized instruction/input template, not by scenario number or duplicate row. With only 14 templates, expect a very small validation/test set; this is transparently reported in the final metrics.

In [ ]:
rng = random.Random(SEED)
shuffled_groups = group_keys.copy(); rng.shuffle(shuffled_groups)
n = len(shuffled_groups)
n_test = max(1, round(n*0.10)); n_val = max(1, round(n*0.10))
while n_test+n_val >= n: n_val = max(1,n_val-1)
train_groups = set(shuffled_groups[:n-n_val-n_test])
val_groups = set(shuffled_groups[n-n_val-n_test:n-n_test])
test_groups = set(shuffled_groups[n-n_test:])
assert not (train_groups&val_groups or train_groups&test_groups or val_groups&test_groups)
parts = {}
for split_name, keys in [('train',train_groups),('validation',val_groups),('test',test_groups)]:
    parts[split_name] = [r for r in records if r['group_key'] in keys]
    print(split_name, 'templates:',len(keys),'examples:',len(parts[split_name]))
if not parts['train'] or not parts['validation'] or not parts['test']:
    raise RuntimeError('One split is empty; increase dataset template diversity.')
print('Template overlap:',len(train_groups&val_groups),len(train_groups&test_groups),len(val_groups&test_groups))
formatted = {}
for split_name, rows in parts.items():
    output_rows=[]
    for r in rows:
        prompt = f"### Instruction:\n{r['instruction']}\n"
        if r['input']: prompt += f"\n### Input:\n{r['input']}\n"
        prompt += "\n### Response:\n"
        output_rows.append({'prompt_text':prompt,'response_text':r['output'],'instruction':r['instruction'],'input':r['input'],'output':r['output']})
    formatted[split_name] = Dataset.from_list(output_rows)
prepared = DatasetDict(formatted)
print(prepared)
prepared.save_to_disk(str(DATASET_EXPORT_DIR))

train templates: 12 examples: 12
validation templates: 1 examples: 1
test templates: 1 examples: 1
Template overlap: 0 0 0
DatasetDict({
    train: Dataset({
        features: ['prompt_text', 'response_text', 'instruction', 'input', 'output'],
        num_rows: 12
    })
    validation: Dataset({
        features: ['prompt_text', 'response_text', 'instruction', 'input', 'output'],
        num_rows: 1
    })
    test: Dataset({
        features: ['prompt_text', 'response_text', 'instruction', 'input', 'output'],
        num_rows: 1
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/12 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1 [00:00<?, ? examples/s]

## 4. Tokenizer and response-only tokenization

The tokenizer is downloaded from Qwen/Qwen3-8B, but the 8B model weights are **not** loaded. Prompt tokens receive label `-100`, so the causal language-model loss is calculated only on the response tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
EOS_ID = tokenizer.eos_token_id

def encode_example(row):
    prompt_ids = tokenizer(row['prompt_text'],add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(row['response_text'],add_special_tokens=False)['input_ids']
    # Preserve the answer; trim prompt from its left if total length exceeds the context.
    answer_ids = answer_ids[:MAX_LENGTH-1]
    prompt_ids = prompt_ids[-max(0,MAX_LENGTH-len(answer_ids)-1):]
    ids = prompt_ids + answer_ids + [EOS_ID]
    labels = [-100]*len(prompt_ids) + answer_ids + [EOS_ID]
    return {'input_ids':ids,'labels':labels}

tokenized = {}
for name, ds in prepared.items():
    tokenized[name] = ds.map(encode_example, remove_columns=ds.column_names, desc=f'Tokenizing {name}')
print({k:len(v) for k,v in tokenized.items()})
if not any(any(t!=-100 for t in row['labels']) for row in tokenized['train']):
    raise RuntimeError('No response labels remain after tokenization.')

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Tokenizing train:   0%|          | 0/12 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/1 [00:00<?, ? examples/s]

{'train': 12, 'validation': 1, 'test': 1}


## 5. Initialize the compact Qwen3 SLM from random weights

This follows the reference notebook's compact dimensions: hidden size 512, 6 layers, 8 attention heads, 2 KV heads, intermediate size 1536, and context 128. `Qwen3ForCausalLM(config)` initializes new weights; no pretrained Qwen3 checkpoint is loaded.

In [ ]:
MODEL_CONFIG = {
    'vocab_size':len(tokenizer), 'hidden_size':512, 'intermediate_size':1536,
    'num_hidden_layers':6, 'num_attention_heads':8, 'num_key_value_heads':2,
    'head_dim':64, 'max_position_embeddings':MAX_LENGTH,
    'rms_norm_eps':1e-6, 'rope_theta':1_000_000.0,
    'attention_dropout':0.0, 'use_cache':False, 'tie_word_embeddings':False,
    'bos_token_id':tokenizer.bos_token_id, 'eos_token_id':tokenizer.eos_token_id,
    'pad_token_id':tokenizer.pad_token_id
}
config = Qwen3Config(**MODEL_CONFIG)
config._attn_implementation = 'sdpa'
model = Qwen3ForCausalLM(config)
model.gradient_checkpointing_enable()
model.config.use_cache = False
model.to(DEVICE)
PARAM_COUNT = sum(p.numel() for p in model.parameters())
print(f'Parameters: {PARAM_COUNT:,} ({PARAM_COUNT/1e6:.2f}M)')
print('Initialization: random from config; pretrained Qwen3 model weights loaded: NO')

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(


Parameters: 173,404,416 (173.40M)
Initialization: random from config; pretrained Qwen3 model weights loaded: NO


## 6. Data loaders, optimizer and precision

FP16 is used on common Colab T4 GPUs; BF16 is selected only when the GPU reports support. Gradient accumulation keeps the effective batch size at 8.

In [ ]:
class EncodedDataset(TorchDataset):
    def __init__(self, ds): self.ds=ds
    def __len__(self): return len(self.ds)
    def __getitem__(self,i): return self.ds[i]

def collate(items):
    L=max(len(x['input_ids']) for x in items)
    ids=[]; labs=[]; masks=[]
    for x in items:
        n=len(x['input_ids']); p=L-n
        ids.append(x['input_ids']+[tokenizer.pad_token_id]*p)
        labs.append(x['labels']+[-100]*p)
        masks.append([1]*n+[0]*p)
    return {'input_ids':torch.tensor(ids,dtype=torch.long),'labels':torch.tensor(labs,dtype=torch.long),'attention_mask':torch.tensor(masks,dtype=torch.long)}

train_loader=DataLoader(EncodedDataset(tokenized['train']),batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate)
val_loader=DataLoader(EncodedDataset(tokenized['validation']),batch_size=1,shuffle=False,collate_fn=collate)
test_loader=DataLoader(EncodedDataset(tokenized['test']),batch_size=1,shuffle=False,collate_fn=collate)
optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,betas=(0.9,0.95),weight_decay=WEIGHT_DECAY)
def lr_lambda(step):
    if step<WARMUP_STEPS: return max(0.05,(step+1)/WARMUP_STEPS)
    p=min(1.0,(step-WARMUP_STEPS)/max(1,MAX_STEPS-WARMUP_STEPS))
    return MIN_LR/LEARNING_RATE+(1-MIN_LR/LEARNING_RATE)*0.5*(1+math.cos(math.pi*p))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda)
use_bf16 = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler=torch.amp.GradScaler('cuda',enabled=(DEVICE.type=='cuda' and amp_dtype==torch.float16))
print('AMP dtype:',amp_dtype,'| GradScaler:',scaler.is_enabled())

AMP dtype: torch.bfloat16 | GradScaler: False


## 7. Checkpoint helpers and validation

In [ ]:
LATEST_PATH=CHECKPOINT_DIR/'latest.pt'; BEST_PATH=CHECKPOINT_DIR/'best.pt'
history=[]; best_val=float('inf'); best_step=0; start_step=0

def save_checkpoint(path, step, val_loss, include_training_state=True):
    payload={'step':step,'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'best_val':best_val,'best_step':best_step,'history':history,'val_loss':val_loss,'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
    if not include_training_state:
        payload = {'step':step,'model':model.state_dict(),'val_loss':val_loss,'best_val':best_val,'best_step':best_step}
    tmp=str(path)+'.tmp'; torch.save(payload,tmp); os.replace(tmp,path)

def evaluate(loader, max_batches=None):
    model.eval(); losses=[]
    with torch.no_grad():
        for i,b in enumerate(loader):
            if max_batches and i>=max_batches: break
            b={k:v.to(DEVICE) for k,v in b.items()}
            with torch.autocast(device_type='cuda',dtype=amp_dtype,enabled=True):
                out=model(**b)
            if torch.isfinite(out.loss): losses.append(float(out.loss.detach().cpu()))
    model.train()
    return float(np.mean(losses)) if losses else float('nan')

if RESUME_FROM_CHECKPOINT:
    if not LATEST_PATH.exists(): raise FileNotFoundError(f'Cannot resume: {LATEST_PATH} not found')
    ckpt=torch.load(LATEST_PATH,map_location='cpu',weights_only=False)
    model.load_state_dict(ckpt['model']); optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler']); scaler.load_state_dict(ckpt.get('scaler',{}))
    start_step=int(ckpt['step']); best_val=float(ckpt['best_val']); best_step=int(ckpt['best_step']); history=ckpt.get('history',[])
    random.setstate(ckpt['python_rng']); np.random.set_state(ckpt['numpy_rng']); torch.set_rng_state(ckpt['torch_rng'])
    if torch.cuda.is_available() and ckpt.get('cuda_rng') is not None: torch.cuda.set_rng_state_all(ckpt['cuda_rng'])
    print('Resumed from optimizer update',start_step)
else:
    # Prevent stale checkpoints from a previous run being mistaken for this fresh run.
    for old_path in [LATEST_PATH, BEST_PATH]:
        if old_path.exists(): old_path.unlink()
    print('Fresh training run. Old checkpoints cleared; new checkpoints will be written to Drive.')

Fresh training run. Old checkpoints cleared; new checkpoints will be written to Drive.


## 8. Train for up to 1,200 optimizer updates

Checkpoints are saved periodically to Drive. If Colab disconnects, rerun the notebook, set `RESUME_FROM_CHECKPOINT = True` in section 1, and rerun from the top. The training cell will restore model, optimizer, scheduler, scaler, step counter, history and random states.

In [ ]:
model.train(); optimizer.zero_grad(set_to_none=True)
train_iter=iter(train_loader)
run_started=time.time()
# A quiet wall-clock safety stop prevents Colab session limits from losing the latest checkpoint.
# The requested optimizer-update ceiling remains 1,200.
QUIET_RUNTIME_GUARD_SECONDS=4080
actual_step=start_step
while actual_step < MAX_STEPS:
    if time.time()-run_started > QUIET_RUNTIME_GUARD_SECONDS: break
    optimizer.zero_grad(set_to_none=True)
    accum_loss=0.0
    for micro in range(GRAD_ACCUM_STEPS):
        try: batch=next(train_iter)
        except StopIteration:
            train_iter=iter(train_loader); batch=next(train_iter)
        batch={k:v.to(DEVICE) for k,v in batch.items()}
        with torch.autocast(device_type='cuda',dtype=amp_dtype,enabled=True):
            out=model(**batch); loss=out.loss/GRAD_ACCUM_STEPS
        if not torch.isfinite(loss): raise RuntimeError('Non-finite training loss; stopping to protect checkpoint.')
        scaler.scale(loss).backward(); accum_loss += float(loss.detach().cpu())
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(),MAX_GRAD_NORM)
    scaler.step(optimizer); scaler.update(); scheduler.step()
    actual_step += 1
    if actual_step % 25 == 0:
        history.append({'step':actual_step,'train_loss':accum_loss,'learning_rate':scheduler.get_last_lr()[0]})
    if actual_step % EVAL_EVERY == 0 or actual_step==MAX_STEPS:
        val_loss=evaluate(val_loader)
        print(f'Update {actual_step}/{MAX_STEPS} | train loss {accum_loss:.4f} | validation loss {val_loss:.4f}')
        if np.isfinite(val_loss) and val_loss < best_val:
            best_val=val_loss; best_step=actual_step; save_checkpoint(BEST_PATH,actual_step,val_loss,include_training_state=False)
        if actual_step % SAVE_EVERY == 0: save_checkpoint(LATEST_PATH,actual_step,val_loss)
    elif actual_step % SAVE_EVERY == 0:
        save_checkpoint(LATEST_PATH,actual_step,float('nan'))
    if actual_step % 25 == 0: gc.collect(); torch.cuda.empty_cache()

# Always save the latest state, including a run stopped before the next evaluation interval.
if actual_step>start_step: save_checkpoint(LATEST_PATH,actual_step,float('nan'))
actual_minutes=(time.time()-run_started)/60
print('Optimizer updates completed:',actual_step,'of',MAX_STEPS)
print('Best validation loss:',best_val,'at update',best_step)
print('Training history points:',len(history))
with open(PROJECT_DIR/'logs'/'training_history.json','w') as f: json.dump(history,f,indent=2)
with open(PROJECT_DIR/'reports'/'training_run.json','w') as f: json.dump({'configured_max_steps':MAX_STEPS,'actual_optimizer_updates':actual_step,'best_step':best_step,'best_validation_loss':best_val,'elapsed_minutes':actual_minutes,'effective_batch_size':BATCH_SIZE*GRAD_ACCUM_STEPS,'seed':SEED},f,indent=2)
if actual_step < MAX_STEPS: print('Run ended before 1,200 updates; latest checkpoint is saved and can be resumed.')

Update 100/1200 | train loss 0.0171 | validation loss 4.8824
Update 200/1200 | train loss 0.0011 | validation loss 5.6515
Update 300/1200 | train loss 0.0001 | validation loss 6.2565
Update 400/1200 | train loss 0.0000 | validation loss 6.7866
Update 500/1200 | train loss 0.0000 | validation loss 7.2631
Update 600/1200 | train loss 0.0000 | validation loss 7.5896
Update 700/1200 | train loss 0.0000 | validation loss 7.6277
Update 800/1200 | train loss 0.0000 | validation loss 7.6151
Update 900/1200 | train loss 0.0000 | validation loss 7.5931
Update 1000/1200 | train loss 0.0000 | validation loss 7.5868
Update 1100/1200 | train loss 0.0000 | validation loss 7.5886
Update 1200/1200 | train loss 0.0000 | validation loss 7.5820
Optimizer updates completed: 1200 of 1200
Best validation loss: 4.882358074188232 at update 100
Training history points: 48


## 9. Restore the best checkpoint and evaluate held-out templates

Metrics are computed on the held-out template groups. With a tiny number of templates, exact match is highly discrete and must be reported with the test sample count.

In [ ]:
if not BEST_PATH.exists():
    # If the run ended before its first evaluation, use latest as a provisional export checkpoint.
    if not LATEST_PATH.exists(): raise RuntimeError('No checkpoint found. Training did not save a checkpoint.')
    ckpt=torch.load(LATEST_PATH,map_location='cpu',weights_only=False)
else: ckpt=torch.load(BEST_PATH,map_location='cpu',weights_only=False)
model.load_state_dict(ckpt['model']); model.eval(); model.config.use_cache=True

def generate_answer(instruction, extra='', max_new_tokens=96):
    prompt=f"### Instruction:\n{clean_instruction(instruction)}\n"
    if extra: prompt+=f"\n### Input:\n{extra}\n"
    prompt+='\n### Response:\n'
    inputs=tokenizer(prompt,return_tensors='pt',add_special_tokens=False).to(DEVICE)
    with torch.no_grad():
        with torch.autocast(device_type='cuda',dtype=amp_dtype,enabled=True):
            out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    new=out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new,skip_special_tokens=True).strip()

def norm_answer(s): return re.sub(r'\s+',' ',str(s).strip()).casefold()
results=[]
for row in prepared['test']:
    pred=generate_answer(row['instruction'],row['input'])
    exp=row['output']
    results.append({'instruction':row['instruction'],'expected':exp,'generated':pred,'exact_match':norm_answer(pred)==norm_answer(exp)})
exact=sum(x['exact_match'] for x in results)/max(1,len(results))
with open(PROJECT_DIR/'reports'/'evaluation_results.json','w') as f: json.dump({'test_templates':len(results),'exact_match_rate':exact,'results':results},f,indent=2)
print('Held-out templates:',len(results),'| exact match:',f'{exact:.2%}')
print('Interpretation: this is a small template-level test, not a broad coding benchmark.')
for x in results: print('\nINSTRUCTION:',x['instruction'],'\nEXPECTED:',x['expected'][:300],'\nGENERATED:',x['generated'][:300],'\nEXACT:',x['exact_match'])

Held-out templates: 1 | exact match: 0.00%
Interpretation: this is a small template-level test, not a broad coding benchmark.

INSTRUCTION: Design a scalable SaaS. Provide a practical implementation plan. 
EXPECTED: Use microservices, API gateway, PostgreSQL, Redis, monitoring, CI/CD, and container orchestration. Include milestones, technology choices, risks, testing strategy, and scaling considerations. 
GENERATED: Use microservices, API gateway, PostgreSQL, PostgreSQL, monitoring, and container, and, technology choices, risks, testing strategy, and scaling considerations. 
EXACT: False


## 10. Export Hugging Face model, tokenizer, dataset and model card

In [ ]:
# Export the best weights in standard Transformers format.
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
model.config.use_cache=True
model.save_pretrained(EXPORT_DIR,safe_serialization=True)
tokenizer.save_pretrained(EXPORT_DIR)
prepared.save_to_disk(str(DATASET_EXPORT_DIR))

metadata={'project':'Vibe Coding Qwen3 SLM','architecture':'Qwen3ForCausalLM','initialization':'random initialization from Qwen3Config; no pretrained Qwen3 weights','tokenizer':TOKENIZER_ID,'source_dataset':DATASET_ID,'parameter_count':PARAM_COUNT,'model_config':MODEL_CONFIG,'max_optimizer_updates':MAX_STEPS,'actual_optimizer_updates':actual_step,'best_checkpoint_step':best_step,'best_validation_loss':best_val,'dataset_audit':audit,'test_templates':len(results),'test_exact_match_rate':exact,'limitations':'Low unique-template diversity; results measure reproduction of held-out templates only and do not establish broad coding generalization.'}
(PROJECT_DIR/'reports'/'model_metadata.json').write_text(json.dumps(metadata,indent=2))
model_card=f"""---
license: apache-2.0
task_categories:
- text-generation
library_name: transformers
---
# Vibe Coding Qwen3 SLM

A compact Qwen3 causal language model initialized from random weights and trained on {DATASET_ID}.

## Important initialization detail
The model was created from `Qwen3Config`; no pretrained Qwen3 model weights were loaded. The tokenizer was downloaded from `{TOKENIZER_ID}`.

## Architecture
- Parameters: {PARAM_COUNT:,}
- Hidden size: 512; layers: 6; attention heads: 8; KV heads: 2; intermediate size: 1536; context: {MAX_LENGTH}.

## Training
- Maximum optimizer updates: {MAX_STEPS}; actual updates: {actual_step}.
- Optimizer: AdamW; warmup + cosine schedule; gradient accumulation; mixed precision; gradient clipping.

## Dataset and evaluation caveat
Raw rows: {audit['raw_rows']}; unique instruction templates: {audit['unique_templates']}; unique instruction/answer pairs: {audit['unique_pairs']}. Splits are grouped by normalized instruction/input template. Test templates: {len(results)}; exact match: {exact:.4f}. The source has low template diversity, so this is a pipeline demonstration and template-reproduction evaluation, not evidence of broad coding ability.

## Use
See the project README and Ollama instructions.
"""
(EXPORT_DIR/'README.md').write_text(model_card)
print('Model export:',EXPORT_DIR,'| files:',len(list(EXPORT_DIR.iterdir())))
print('Prepared dataset:',DATASET_EXPORT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/12 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1 [00:00<?, ? examples/s]

Model export: /content/drive/MyDrive/Vibe_Coding_Qwen3_SLM_Project/hf_model | files: 7
Prepared dataset: /content/drive/MyDrive/Vibe_Coding_Qwen3_SLM_Project/hf_dataset


## 11. Optional Hugging Face upload

Set your own namespace in section 1 and set `UPLOAD_TO_HUGGING_FACE = True`. The login flow asks for a Hugging Face write token without placing it in notebook source. The model and prepared dataset are uploaded to separate repositories.

In [ ]:

# ============================================================
# HUGGING FACE: UPLOAD TRAINED SLM + PREPARED DATASET
# ============================================================

if UPLOAD_TO_HUGGING_FACE:

    from pathlib import Path
    from huggingface_hub import HfApi, notebook_login

    print("=" * 60)
    print("HUGGING FACE UPLOAD")
    print("=" * 60)

    # 1. Validate repository IDs
    def validate_repo_id(repo_id, name):
        if not isinstance(repo_id, str) or repo_id.count("/") != 1:
            raise ValueError(
                f"{name} must use the format username/repository-name. "
                f"Current value: {repo_id}"
            )

        namespace, repo_name = repo_id.split("/")

        if not namespace.strip() or not repo_name.strip():
            raise ValueError(f"Invalid {name}: {repo_id}")

    validate_repo_id(MODEL_REPO_ID, "MODEL_REPO_ID")
    validate_repo_id(DATASET_REPO_ID, "DATASET_REPO_ID")

    # 2. Check export directories
    model_dir = Path(EXPORT_DIR)
    dataset_dir = Path(DATASET_EXPORT_DIR)

    if not model_dir.is_dir():
        raise FileNotFoundError(
            f"Trained model folder not found: {model_dir}"
        )

    if not dataset_dir.is_dir():
        raise FileNotFoundError(
            f"Prepared dataset folder not found: {dataset_dir}"
        )

    model_files = [p for p in model_dir.rglob("*") if p.is_file()]
    dataset_files = [p for p in dataset_dir.rglob("*") if p.is_file()]

    if not model_files:
        raise RuntimeError("The model export folder is empty.")

    if not dataset_files:
        raise RuntimeError("The dataset export folder is empty.")

    print(f"\nModel repository: {MODEL_REPO_ID}")
    print(f"Dataset repository: {DATASET_REPO_ID}")
    print(f"Model files found: {len(model_files)}")
    print(f"Dataset files found: {len(dataset_files)}")

    # 3. Authenticate with Hugging Face
    print("\nPlease authenticate with your Hugging Face account.")
    print("Use an access token with WRITE permission.")

    notebook_login()

    api = HfApi()

    # Verify authentication
    user_info = api.whoami()
    print(f"\nSuccessfully logged in as: {user_info['name']}")

    # 4. Create model repository
    print("\nCreating/verifying model repository...")

    api.create_repo(
        repo_id=MODEL_REPO_ID,
        repo_type="model",
        exist_ok=True
    )

    # 5. Create dataset repository
    print("Creating/verifying dataset repository...")

    api.create_repo(
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
        exist_ok=True
    )

    # 6. Upload trained model
    print("\nUploading trained SLM...")
    print("Please do not interrupt the upload.")

    api.upload_folder(
        folder_path=str(model_dir),
        repo_id=MODEL_REPO_ID,
        repo_type="model",
        commit_message="Upload trained Vibe Coding Qwen3 SLM"
    )

    print("\nMODEL UPLOAD SUCCESSFUL!")
    print(f"https://huggingface.co/{MODEL_REPO_ID}")

    # 7. Upload prepared dataset
    print("\nUploading prepared dataset...")

    api.upload_folder(
        folder_path=str(dataset_dir),
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
        commit_message="Upload prepared Vibe Coding instruction dataset"
    )

    print("\nDATASET UPLOAD SUCCESSFUL!")
    print(f"https://huggingface.co/datasets/{DATASET_REPO_ID}")

    # 8. Final confirmation
    print("\n" + "=" * 60)
    print("ALL HUGGING FACE UPLOADS COMPLETED SUCCESSFULLY!")
    print("=" * 60)

    print("\nMODEL:")
    print(f"https://huggingface.co/{MODEL_REPO_ID}")

    print("\nDATASET:")
    print(f"https://huggingface.co/datasets/{DATASET_REPO_ID}")

else:
    print("Hugging Face upload skipped.")
    print("Set UPLOAD_TO_HUGGING_FACE = True in Section 1 to enable.")


HUGGING FACE UPLOAD

Model repository: Aslam7/Vibe-Coding-Qwen3-SLM
Dataset repository: Aslam7/Vibe-Coding-Instruct-Prepared
Model files found: 7
Dataset files found: 10

Please authenticate with your Hugging Face account.
Use an access token with WRITE permission.



Successfully logged in as: Aslam7

Creating/verifying model repository...
Creating/verifying dataset repository...

Uploading trained SLM...
Please do not interrupt the upload.

MODEL UPLOAD SUCCESSFUL!
https://huggingface.co/Aslam7/Vibe-Coding-Qwen3-SLM

Uploading prepared dataset...

DATASET UPLOAD SUCCESSFUL!
https://huggingface.co/datasets/Aslam7/Vibe-Coding-Instruct-Prepared

ALL HUGGING FACE UPLOADS COMPLETED SUCCESSFULLY!

MODEL:
https://huggingface.co/Aslam7/Vibe-Coding-Qwen3-SLM

DATASET:
https://huggingface.co/datasets/Aslam7/Vibe-Coding-Instruct-Prepared


## 12. Convert to GGUF and quantize for Ollama

Uses the official `llama.cpp` converter without patching `config.json`. Qwen3 is supported by current llama.cpp versions; converter support is version-dependent, so a failure should be recorded rather than worked around by falsifying the model type. Q4_K_M output is then used for local Ollama inference.

In [ ]:

# ============================================================
# QWEN3 TOKENIZER REPAIR
# Fix missing tokenizer assets and Transformers compatibility
# ============================================================

import json
import shutil
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

print("=" * 60)
print("REPAIRING QWEN3 TOKENIZER")
print("=" * 60)

# Use the original tokenizer from Section 1
TOKENIZER_ID = globals().get("TOKENIZER_ID", "Qwen/Qwen3-8B")

EXPORT_DIR = Path(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Download original tokenizer assets
print("\nDownloading original Qwen3 tokenizer...")

tokenizer_source = Path(snapshot_download(
    repo_id=TOKENIZER_ID,
    allow_patterns=[
        "tokenizer.json",
        "tokenizer_config.json",
        "vocab.json",
        "merges.txt",
        "added_tokens.json",
        "special_tokens_map.json",
        "chat_template.jinja"
    ]
))

# 2. Copy original tokenizer files into trained model folder
tokenizer_names = [
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
    "added_tokens.json",
    "special_tokens_map.json",
    "chat_template.jinja"
]

copied = []

for filename in tokenizer_names:
    source = tokenizer_source / filename

    if source.is_file():
        shutil.copy2(source, EXPORT_DIR / filename)
        copied.append(filename)

print("Tokenizer files copied:", copied)

if not (EXPORT_DIR / "tokenizer.json").exists():
    raise FileNotFoundError("Original tokenizer.json was not downloaded.")

if not (EXPORT_DIR / "tokenizer_config.json").exists():
    raise FileNotFoundError("Original tokenizer_config.json is missing.")

# 3. Fix incompatible extra_special_tokens field
config_file = EXPORT_DIR / "tokenizer_config.json"

with open(config_file, "r", encoding="utf-8") as f:
    tokenizer_config = json.load(f)

extra = tokenizer_config.get("extra_special_tokens")

if isinstance(extra, list):
    print("Fixing extra_special_tokens: list -> dictionary")
    tokenizer_config["extra_special_tokens"] = {}

elif extra is not None and not isinstance(extra, dict):
    print("Removing incompatible extra_special_tokens field")
    tokenizer_config["extra_special_tokens"] = {}

with open(config_file, "w", encoding="utf-8") as f:
    json.dump(tokenizer_config, f, indent=2, ensure_ascii=False)

# 4. Validate tokenizer locally
print("\nValidating tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    str(EXPORT_DIR),
    use_fast=True,
    local_files_only=True
)

print("Tokenizer loaded successfully!")
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Tokenizer vocabulary:", tokenizer.vocab_size)
print("Total tokens:", len(tokenizer))


# 5. Correctly validate tokenizer vocabulary

with open(EXPORT_DIR / "config.json", "r") as f:
    model_config = json.load(f)

expected_vocab = model_config.get("vocab_size")
actual_vocab = len(tokenizer)

print("\nVocabulary validation:")
print("Model vocabulary:", expected_vocab)
print("Tokenizer base vocabulary:", tokenizer.vocab_size)
print("Tokenizer total vocabulary:", actual_vocab)

if expected_vocab != actual_vocab:
    raise ValueError(
        f"Actual vocabulary mismatch: "
        f"Model={expected_vocab}, Tokenizer={actual_vocab}. "
        "Do not continue with GGUF conversion."
    )

print("\nSUCCESS: Model and tokenizer vocabularies match!")
print("Tokenizer validation completed successfully.")


print("\n" + "=" * 60)
print("TOKENIZER REPAIR SUCCESSFUL")
print("=" * 60)
print("You can now run the GGUF conversion cell.")


REPAIRING QWEN3 TOKENIZER



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Tokenizer files copied: ['tokenizer.json', 'tokenizer_config.json', 'vocab.json', 'merges.txt']

Validating tokenizer...
Tokenizer loaded successfully!
Tokenizer class: Qwen2Tokenizer
Tokenizer vocabulary: 151643
Total tokens: 151669

Vocabulary validation:
Model vocabulary: 151669
Tokenizer base vocabulary: 151643
Tokenizer total vocabulary: 151669

SUCCESS: Model and tokenizer vocabularies match!
Tokenizer validation completed successfully.

TOKENIZER REPAIR SUCCESSFUL
You can now run the GGUF conversion cell.


In [ ]:

# ============================================================
# FINAL QWEN3 GGUF CONVERSION AND QUANTIZATION
# ============================================================

import sys
import subprocess
from pathlib import Path

GGUF_DIR = Path(PROJECT_DIR) / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)

LLAMA_CPP = Path("/content/llama.cpp")
EXPORT_DIR = Path(EXPORT_DIR)

def run_command(name, command):

    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    print(result.stdout[-12000:])

    if result.returncode != 0:
        raise RuntimeError(
            f"{name} failed. See the output above."
        )

    print(name, "SUCCESS")


if CONVERT_TO_GGUF:

    # 1. Check repaired tokenizer
    required = [
        "config.json",
        "model.safetensors",
        "tokenizer.json",
        "tokenizer_config.json",
        "vocab.json",
        "merges.txt"
    ]

    missing = [
        f for f in required
        if not (EXPORT_DIR / f).exists()
    ]

    if missing:
        raise FileNotFoundError(
            f"Missing required model files: {missing}. "
            "Run the tokenizer repair cell first."
        )

    # 2. Check llama.cpp
    if not (LLAMA_CPP / ".git").exists():

        run_command(
            "Cloning llama.cpp",
            [
                "git", "clone", "--depth", "1",
                "https://github.com/ggml-org/llama.cpp",
                str(LLAMA_CPP)
            ]
        )

    converter = LLAMA_CPP / "convert_hf_to_gguf.py"

    if not converter.exists():
        raise FileNotFoundError("GGUF converter missing.")

    # 3. Convert model to F16
    f16_path = GGUF_DIR / "vibe-coding-qwen3-f16.gguf"

    if f16_path.exists():
        f16_path.unlink()

    run_command(
        "Converting trained Qwen3 model to F16 GGUF",
        [
            sys.executable,
            str(converter),
            str(EXPORT_DIR),
            "--outfile", str(f16_path),
            "--outtype", "f16"
        ]
    )

    if not f16_path.exists() or f16_path.stat().st_size == 0:
        raise RuntimeError("F16 GGUF generation failed.")

    with open(f16_path, "rb") as f:
        if f.read(4) != b"GGUF":
            raise RuntimeError("Invalid F16 GGUF header.")

    # 4. Build CPU quantizer
    build = LLAMA_CPP / "build"
    build.mkdir(exist_ok=True)

    run_command(
        "Configuring llama.cpp",
        [
            "cmake", "-S", str(LLAMA_CPP),
            "-B", str(build),
            "-DGGML_CUDA=OFF",
            "-DLLAMA_BUILD_TESTS=OFF",
            "-DLLAMA_BUILD_EXAMPLES=OFF"
        ]
    )

    run_command(
        "Building llama-quantize",
        [
            "cmake", "--build", str(build),
            "--config", "Release",
            "-j", "2",
            "--target", "llama-quantize"
        ]
    )

    quantizer = build / "bin" / "llama-quantize"

    if not quantizer.exists():
        raise FileNotFoundError("Quantizer executable not found.")

    # 5. Generate quantized models
    quantizations = {
        "Q8_0": "vibe-coding-qwen3-Q8_0.gguf",
        "Q5_K_M": "vibe-coding-qwen3-Q5_K_M.gguf",
        "Q4_K_M": "vibe-coding-qwen3-Q4_K_M.gguf"
    }

    for quant_type, filename in quantizations.items():

        output_path = GGUF_DIR / filename

        if output_path.exists():
            output_path.unlink()

        run_command(
            f"Generating {quant_type}",
            [
                str(quantizer),
                str(f16_path),
                str(output_path),
                quant_type
            ]
        )

        if not output_path.exists() or output_path.stat().st_size == 0:
            raise RuntimeError(f"{quant_type} generation failed.")

        with open(output_path, "rb") as f:
            if f.read(4) != b"GGUF":
                raise RuntimeError(f"Invalid {quant_type} GGUF.")

    # 6. Final report
    print("\n" + "=" * 60)
    print("ALL GGUF FILES GENERATED SUCCESSFULLY")
    print("=" * 60)

    for file in sorted(GGUF_DIR.glob("*.gguf")):
        print(
            file.name,
            ":", round(file.stat().st_size / 1024**2, 2),
            "MB"
        )

    print("\nOutput directory:", GGUF_DIR)

else:
    print("GGUF conversion skipped.")



Converting trained Qwen3 model to F16 GGUF
riter:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,            torch.float32 --> F16, shape = {512, 151669}
INFO:hf-to-gguf:token_embd.weight,        torch.float32 --> F16, shape = {512, 151669}
INFO:hf-to-gguf:blk.0.attn_norm.weight,   torch.float32 --> F32, shape = {512}
INFO:hf-to-gguf:blk.0.ffn_down.weight,    torch.float32 --> F16, shape = {1536, 512}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,    torch.float32 --> F16, shape = {512, 1536}
INFO:hf-to-gguf:blk.0.ffn_up.weight,      torch.float32 --> F16, shape = {512, 1536}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,    torch.float32 --> F32, shape = {512}
INFO:hf-to-gguf:blk.0.attn_k_norm.weight, torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:blk.0.attn_k.weight,      torch.float32 --> F16, shape = {512, 128}
INFO:hf-to-gguf:blk.0.attn_output.weight, torch.float32 --> F16, shape = {512, 512}
INFO:hf-to-gguf:blk.0.attn_q_norm.w

## 13. Create Ollama Modelfile and local demo instructions

The GGUF file and Modelfile must be placed in the same folder on your laptop. Ollama must be installed locally. Colab cannot install the model into your laptop's Ollama instance directly.

In [ ]:
q4_path=GGUF_DIR/'vibe-coding-qwen3-Q4_K_M.gguf'
modelfile_text='''FROM ./vibe-coding-qwen3-Q4_K_M.gguf
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER num_ctx 128
PARAMETER num_predict 128
'''
(GGUF_DIR/'Modelfile').write_text(modelfile_text)
print('Modelfile:',GGUF_DIR/'Modelfile')
print('Required laptop commands:')
print('  ollama create vibe-coding-qwen3 -f Modelfile')
print('  ollama run vibe-coding-qwen3')
print('Try: Create a simple Python function that checks whether a number is prime.')

Modelfile: /content/drive/MyDrive/Vibe_Coding_Qwen3_SLM_Project/gguf/Modelfile
Required laptop commands:
  ollama create vibe-coding-qwen3 -f Modelfile
  ollama run vibe-coding-qwen3
Try: Create a simple Python function that checks whether a number is prime.


## 14. Build a clean Drive release folder and ZIP

The release is assembled only after all previous stages. Missing optional GGUF artifacts are clearly omitted; no placeholder files are created.

In [ ]:
RELEASE=PROJECT_DIR/'release';
if RELEASE.exists(): shutil.rmtree(RELEASE)
for d in ['model','dataset','checkpoints','reports','gguf','notebook','github_repo']: (RELEASE/d).mkdir(parents=True,exist_ok=True)
shutil.copytree(EXPORT_DIR,RELEASE/'model',dirs_exist_ok=True)
shutil.copytree(DATASET_EXPORT_DIR,RELEASE/'dataset',dirs_exist_ok=True)
for p in CHECKPOINT_DIR.glob('*.pt'): shutil.copy2(p,RELEASE/'checkpoints'/p.name)
for p in (PROJECT_DIR/'reports').glob('*'):
    if p.is_file(): shutil.copy2(p,RELEASE/'reports'/p.name)
for p in (PROJECT_DIR/'logs').glob('*'):
    if p.is_file(): shutil.copy2(p,RELEASE/'reports'/p.name)
for p in GGUF_DIR.glob('*'):
    if p.is_file(): shutil.copy2(p,RELEASE/'gguf'/p.name)
# Copy notebook from the Colab working directory if saved there; otherwise project files still remain complete.
print('Release contents:')
for root,dirs,files in os.walk(RELEASE):
    for fn in files: print(Path(root).relative_to(RELEASE)/fn)
zip_path=PROJECT_DIR/'Vibe_Coding_Qwen3_SLM_Release.zip'
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in RELEASE.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(RELEASE))
print('Release ZIP:',zip_path,'| MB:',round(zip_path.stat().st_size/1024**2,2))

Release contents:
model/config.json
model/generation_config.json
model/model.safetensors
model/chat_template.jinja
model/tokenizer_config.json
model/tokenizer.json
model/README.md
model/vocab.json
model/merges.txt
dataset/dataset_dict.json
dataset/train/data-00000-of-00001.arrow
dataset/train/state.json
dataset/train/dataset_info.json
dataset/validation/data-00000-of-00001.arrow
dataset/validation/state.json
dataset/validation/dataset_info.json
dataset/test/data-00000-of-00001.arrow
dataset/test/state.json
dataset/test/dataset_info.json
checkpoints/best.pt
checkpoints/latest.pt
reports/dataset_audit.json
reports/dataset_audit.csv
reports/training_run.json
reports/evaluation_results.json
reports/model_metadata.json
reports/training_history.json
gguf/vibe-coding-qwen3-f16.gguf
gguf/vibe-coding-qwen3-Q8_0.gguf
gguf/vibe-coding-qwen3-Q5_K_M.gguf
gguf/vibe-coding-qwen3-Q4_K_M.gguf
gguf/Modelfile
Release ZIP: /content/drive/MyDrive/Vibe_Coding_Qwen3_SLM_Project/Vibe_Coding_Qwen3_SLM_Release.

## 15. Final validation report

This cell verifies required local artifacts and reports any steps that still require account credentials or a laptop-side Ollama test.

In [ ]:
checks={
 'best checkpoint':BEST_PATH.exists(), 'latest checkpoint':LATEST_PATH.exists(),
 'HF model config':(EXPORT_DIR/'config.json').exists(), 'model weights':any(EXPORT_DIR.glob('*.safetensors')),
 'tokenizer':(EXPORT_DIR/'tokenizer.json').exists() or (EXPORT_DIR/'tokenizer_config.json').exists(),
 'prepared dataset':DATASET_EXPORT_DIR.exists(), 'evaluation report':(PROJECT_DIR/'reports'/'evaluation_results.json').exists(),
 'Ollama GGUF':(GGUF_DIR/'vibe-coding-qwen3-Q4_K_M.gguf').exists(), 'Ollama Modelfile':(GGUF_DIR/'Modelfile').exists(), 'release ZIP':zip_path.exists()
}
for k,v in checks.items(): print(('OK  ' if v else 'TODO'),k)
print('Updates completed:',actual_step,'/',MAX_STEPS)
print('Dataset templates:',audit['unique_templates'],'| held-out test templates:',len(results))
print('HF upload enabled:',UPLOAD_TO_HUGGING_FACE,'| GitHub push enabled:',PUSH_TO_GITHUB)
print('Remember: Ollama must be tested on the presentation laptop.')

OK   best checkpoint
OK   latest checkpoint
OK   HF model config
OK   model weights
OK   tokenizer
OK   prepared dataset
OK   evaluation report
OK   Ollama GGUF
OK   Ollama Modelfile
OK   release ZIP
Updates completed: 1200 / 1200
Dataset templates: 14 | held-out test templates: 1
HF upload enabled: True | GitHub push enabled: False
Remember: Ollama must be tested on the presentation laptop.


after this all troubleshoot

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Vibe_Coding_Qwen3_SLM_Project"
)

print("Project folder exists:", PROJECT_DIR.exists())

if PROJECT_DIR.exists():
    print("\nFiles and folders:")
    for item in sorted(PROJECT_DIR.iterdir()):
        print(item.name, "—", "Folder" if item.is_dir() else "File")
else:
    print("Project folder not found at this location.")
    print("\nMatching folders in MyDrive:")
    drive_root = Path("/content/drive/MyDrive")
    for item in drive_root.iterdir():
        if "Vibe_Coding" in item.name:
            print(item)


Project folder exists: True

Files and folders:
Vibe_Coding_Qwen3_SLM_Release.zip — File
checkpoints — Folder
gguf — Folder
github_repo — Folder
hf_dataset — Folder
hf_model — Folder
logs — Folder
release — Folder
reports — Folder


In [ ]:

from pathlib import Path

folders = [
    "checkpoints",
    "hf_dataset",
    "hf_model",
    "reports",
    "logs",
    "gguf"
]

for folder in folders:
    path = PROJECT_DIR / folder
    print(f"\n{'='*55}")
    print(f"FOLDER: {folder}")
    print('='*55)

    if not path.exists():
        print("Not found")
        continue

    for item in sorted(path.rglob("*")):
        if item.is_file():
            size_mb = item.stat().st_size / (1024 * 1024)
            print(f"{item.relative_to(path)}  ({size_mb:.2f} MB)")
        elif item.is_dir():
            print(f"[DIR] {item.relative_to(path)}")



FOLDER: checkpoints
best.pt  (661.51 MB)
latest.pt  (1984.55 MB)

FOLDER: hf_dataset
dataset_dict.json  (0.00 MB)
[DIR] test
test/data-00000-of-00001.arrow  (0.00 MB)
test/dataset_info.json  (0.00 MB)
test/state.json  (0.00 MB)
[DIR] train
train/data-00000-of-00001.arrow  (0.01 MB)
train/dataset_info.json  (0.00 MB)
train/state.json  (0.00 MB)
[DIR] validation
validation/data-00000-of-00001.arrow  (0.00 MB)
validation/dataset_info.json  (0.00 MB)
validation/state.json  (0.00 MB)

FOLDER: hf_model
README.md  (0.00 MB)
chat_template.jinja  (0.00 MB)
config.json  (0.00 MB)
generation_config.json  (0.00 MB)
merges.txt  (1.59 MB)
model.safetensors  (661.49 MB)
tokenizer.json  (10.89 MB)
tokenizer_config.json  (0.01 MB)
vocab.json  (2.65 MB)

FOLDER: reports
dataset_audit.csv  (0.00 MB)
dataset_audit.json  (0.00 MB)
evaluation_results.json  (0.00 MB)
model_metadata.json  (0.00 MB)
training_run.json  (0.00 MB)

FOLDER: logs
training_history.json  (0.01 MB)

FOLDER: gguf
Modelfile  (0.00 MB)


In [ ]:
import json
import pandas as pd
from pathlib import Path

# Confirm project directory
print("PROJECT_DIR:", PROJECT_DIR)

# 1. Inspect reports
report_files = [
    "dataset_audit.json",
    "evaluation_results.json",
    "model_metadata.json",
    "training_run.json"
]

for filename in report_files:
    path = PROJECT_DIR / "reports" / filename
    print(f"\n{'='*65}")
    print(f"REPORT: {filename}")
    print('='*65)

    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(json.dumps(data, indent=2)[:12000])
    else:
        print("File not found")

# 2. Inspect training history
history_path = PROJECT_DIR / "logs" / "training_history.json"

print(f"\n{'='*65}")
print("TRAINING HISTORY")
print('='*65)

if history_path.exists():
    with open(history_path, "r", encoding="utf-8") as f:
        history = json.load(f)

    print(json.dumps(history, indent=2)[:12000])

# 3. Inspect dataset audit CSV
csv_path = PROJECT_DIR / "reports" / "dataset_audit.csv"

print(f"\n{'='*65}")
print("DATASET AUDIT CSV")
print('='*65)

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("Shape:", df.shape)
    print(df.head(20).to_string(index=False))

# 4. Inspect model configuration
config_path = PROJECT_DIR / "hf_model" / "config.json"

print(f"\n{'='*65}")
print("MODEL CONFIGURATION")
print('='*65)

if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)
    print(json.dumps(config, indent=2))

# 5. Inspect dataset examples
print(f"\n{'='*65}")
print("PREPARED DATASET EXAMPLES")
print('='*65)

try:
    from datasets import load_from_disk

    dataset = load_from_disk(str(PROJECT_DIR / "hf_dataset"))

    for split in dataset:
        print(f"\n{split.upper()}: {len(dataset[split])} rows")
        for i, example in enumerate(dataset[split]):
            if i >= 2:
                break
            print(f"\nExample {i+1}:")
            print(example)

except Exception as e:
    print("Dataset inspection issue:", repr(e))

PROJECT_DIR: /content/drive/MyDrive/Vibe_Coding_Qwen3_SLM_Project

REPORT: dataset_audit.json
{
  "dataset_id": "CodeDevX/Vibe-Coding-Instruct",
  "raw_rows": 1100000,
  "unique_pairs": 14,
  "unique_templates": 14,
  "unique_outputs": 14,
  "deduplication": "exact normalized instruction/input/output",
  "split_unit": "normalized instruction + input template"
}

REPORT: evaluation_results.json
{
  "test_templates": 1,
  "exact_match_rate": 0.0,
  "results": [
    {
      "instruction": "Design a scalable SaaS. Provide a practical implementation plan.",
      "expected": "Use microservices, API gateway, PostgreSQL, Redis, monitoring, CI/CD, and container orchestration. Include milestones, technology choices, risks, testing strategy, and scaling considerations.",
      "generated": "Use microservices, API gateway, PostgreSQL, PostgreSQL, monitoring, and container, and, technology choices, risks, testing strategy, and scaling considerations.",
      "exact_match": false
    }
  ]
}

REPOR

In [ ]:
from datasets import load_dataset

DATASET_ID = "CodeDevX/Vibe-Coding-Instruct"

print("Loading original dataset...")
raw_dataset = load_dataset(DATASET_ID)

print("\nDATASET SPLITS")
print(raw_dataset)

for split_name, split_data in raw_dataset.items():
    print("\n" + "=" * 70)
    print("SPLIT:", split_name)
    print("ROWS:", len(split_data))
    print("COLUMNS:", split_data.column_names)
    print("FEATURES:", split_data.features)

    print("\nFIRST 5 RAW EXAMPLES")
    for i in range(min(5, len(split_data))):
        print(f"\n--- Example {i+1} ---")
        print(split_data[i])

Loading original dataset...


README.md:   0%|          | 0.00/176 [00:00<?, ?B/s]

dataset.jsonl: reconstructing file:   0%|          |  0.00B /  459MB            

dataset.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1100000 [00:00<?, ? examples/s]


DATASET SPLITS
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'prompt'],
        num_rows: 1100000
    })
})

SPLIT: train
ROWS: 1100000
COLUMNS: ['instruction', 'input', 'output', 'prompt']
FEATURES: {'instruction': Value('string'), 'input': Value('string'), 'output': Value('string'), 'prompt': Value('string')}

FIRST 5 RAW EXAMPLES

--- Example 1 ---
{'instruction': 'Create a coding assistant. Scenario 0: Provide a practical implementation plan.', 'input': '', 'output': 'Use Electron, Monaco Editor, local LLM integration, chat memory, and project indexing. Include milestones, technology choices, risks, testing strategy, and scaling considerations.', 'prompt': '### Instruction:\nCreate a coding assistant. Scenario 0: Provide a practical implementation plan.\n\n### Input:\n\n\n### Response:\nUse Electron, Monaco Editor, local LLM integration, chat memory, and project indexing. Include milestones, technology choices, risks, testing strategy, and

In [ ]:
import re
from collections import Counter

data = raw_dataset["train"]

raw_instructions = set()
normalized_instructions = set()
raw_outputs = set()
normalized_outputs = set()
normalized_pairs = set()

input_nonempty = 0
prompt_mismatches = 0

output_frequency = Counter()
template_frequency = Counter()

def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r"scenario\s+\d+\s*:", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

for row in data:
    instruction = row["instruction"] or ""
    input_text = row["input"] or ""
    output = row["output"] or ""

    raw_instructions.add(instruction)
    raw_outputs.add(output)

    norm_instruction = normalize(instruction)
    norm_output = normalize(output)
    norm_input = normalize(input_text)

    normalized_instructions.add(norm_instruction)
    normalized_outputs.add(norm_output)

    normalized_pairs.add(
        (norm_instruction, norm_input, norm_output)
    )

    template_frequency[norm_instruction] += 1
    output_frequency[norm_output] += 1

    if input_text.strip():
        input_nonempty += 1

    expected_prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n{output}"
    )

    if row["prompt"].strip() != expected_prompt.strip():
        prompt_mismatches += 1

print("=" * 65)
print("DATASET DIVERSITY DIAGNOSTIC")
print("=" * 65)

print("Total rows:", len(data))
print("Unique raw instructions:", len(raw_instructions))
print("Unique normalized instructions:", len(normalized_instructions))
print("Unique raw outputs:", len(raw_outputs))
print("Unique normalized outputs:", len(normalized_outputs))
print("Unique normalized instruction/input/output combinations:",
      len(normalized_pairs))

print("\nINPUT ANALYSIS")
print("Rows with non-empty input:", input_nonempty)
print("Rows with empty input:", len(data) - input_nonempty)

print("\nPROMPT CONSISTENCY")
print("Prompt mismatches:", prompt_mismatches)

print("\nTOP 15 MOST FREQUENT INSTRUCTION TEMPLATES")
for text, count in template_frequency.most_common(15):
    print(f"{count:,} rows | {text}")

print("\nTOP 15 MOST FREQUENT OUTPUTS")
for text, count in output_frequency.most_common(15):
    print(f"{count:,} rows | {text[:250]}")

print("\nSAMPLE RECORDS WITH THE SAME NORMALIZED INSTRUCTION")
target = normalize(data[0]["instruction"])
shown = 0

for row in data:
    if normalize(row["instruction"]) == target:
        print("\nInstruction:", row["instruction"])
        print("Output:", row["output"])
        shown += 1
        if shown == 5:
            break

DATASET DIVERSITY DIAGNOSTIC
Total rows: 1100000
Unique raw instructions: 1100000
Unique normalized instructions: 1000010
Unique raw outputs: 14
Unique normalized outputs: 14
Unique normalized instruction/input/output combinations: 1000010

INPUT ANALYSIS
Rows with non-empty input: 0
Rows with empty input: 1100000

PROMPT CONSISTENCY
Prompt mismatches: 0

TOP 15 MOST FREQUENT INSTRUCTION TEMPLATES
10,174 rows | integrate local llms. provide a practical implementation plan.
10,147 rows | react app re-renders infinitely. provide a practical implementation plan.
10,111 rows | design a scalable saas. provide a practical implementation plan.
10,023 rows | create a coding assistant. provide a practical implementation plan.
10,012 rows | deploy an ai application. provide a practical implementation plan.
9,985 rows | deploy a mern application. provide a practical implementation plan.
9,960 rows | api returns 500 errors. provide a practical implementation plan.
9,949 rows | add ai chat to an ap

In [ ]:
# Run this in a new code cell in your existing Colab notebook AFTER mounting Drive.
# It reads the small prepared dataset already saved by your notebook; it does not
# reload the 1.1-million-row source dataset or run training again.

import json
from pathlib import Path
from datasets import load_from_disk
from google.colab import files

PROJECT_DIR = Path('/content/drive/MyDrive/Vibe_Coding_Qwen3_SLM_Project')
PREPARED_DIR = PROJECT_DIR / 'hf_dataset'
if not PREPARED_DIR.exists():
    raise FileNotFoundError(f'Prepared dataset folder not found: {PREPARED_DIR}')

prepared = load_from_disk(str(PREPARED_DIR))
examples = []
seen = set()
for split_name in prepared:
    for row in prepared[split_name]:
        item = {
            'instruction': str(row.get('instruction', '')).strip(),
            'input': str(row.get('input', '')).strip(),
            'output': str(row.get('output', '')).strip(),
        }
        key = (item['instruction'].casefold(), item['input'].casefold(), item['output'].casefold())
        if item['instruction'] and item['output'] and key not in seen:
            seen.add(key)
            examples.append(item)

out_path = PROJECT_DIR / 'vibe_dataset_examples.json'
out_path.write_text(json.dumps(examples, indent=2, ensure_ascii=False), encoding='utf-8')
print('Exported examples:', len(examples))
print('File:', out_path)
print('Size (KB):', round(out_path.stat().st_size / 1024, 2))
files.download(str(out_path))


Exported examples: 14
File: /content/drive/MyDrive/Vibe_Coding_Qwen3_SLM_Project/vibe_dataset_examples.json
Size (KB): 3.86


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>